# 4. Три признака числа тем

Для каждой обученной модели вычисляются (раздел 4 статьи):

* **энергия** E — у RTM от начального состояния (порог p_bg, масса тех же ячеек в фоне, формула 13), у LDA, ProdLDA
  и ETM — от 1/V; излом — стык двух отрезков по ln T (формулы 14–15);
* **воспроизводимость** — медиана дивергенции Йенсена–Шеннона (биты) тем, сопоставленных венгерским алгоритмом во всех
  трёх парах запусков, по полным темам и по нормированному лифту;
* **силуэт** топ-слов по векторам word2vec (формулы 28–33);
* на синтетике — **число восстановленных истинных тем** (JS < 0.1 при сопоставлении один к одному).

Результат — `results/curves.csv`, `results/stability.csv`, `results/recovery.csv`. В репозитории эти файлы уже
лежат (посчитаны по моделям статьи), поэтому блокнот 05 работает и без переобучения.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

In [2]:
import numpy as np
from rtm import criteria as cr
from rtm.data import DATA, load_bow, background, word_vectors
from rtm.scan import load_topics, model_dir

# пример на одной модели из блокнотов 02–03
CORPUS, T = "lenta_E2", 10
X, vocab = load_bow(DATA / CORPUS)
p_bg, E = background(X), word_vectors(CORPUS)
for m in ("RTM", "LDA", "ProdLDA", "ETM"):
    phi = load_topics(m, model_dir(m, CORPUS, 0, T), vocab)
    if phi is not None:
        print(f"{m:8s} E = {cr.energy(phi, m, p_bg):.4f}   силуэт = {cr.silhouette(phi, p_bg, E):.4f}")

RTM      E = -1.0859   силуэт = 0.3235
LDA      E = 0.0512   силуэт = 0.2892
ProdLDA  E = 0.4470   силуэт = 0.4011
ETM      E = 0.0984   силуэт = 0.2112


## Все модели перебора

Пересчёт по всем моделям в `models/` (после блокнотов 02–03 с `RUN_SCAN = True`) занимает около часа.
Чтобы не перезаписать поставляемые результаты, по умолчанию он пишет в `results/recomputed/`.

In [3]:
from rtm.scan import compute_results

RUN = False
if RUN:
    curves, stability, recovery = compute_results(out_dir=ROOT / "results" / "recomputed")